# 📗 자연어 처리 — 텍스트 전처리

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지금까지 다룬 데이터는 **표(정형 데이터)** 였습니다. 행과 열이 있고, 값은 숫자나 정해진 범주였죠. 이번 시간부터는 **글(비정형 텍스트)** 을 다룹니다. 사람이 쓴 리뷰 한 줄에는 오타·특수문자·`ㅋㅋㅋ`·줄임말이 뒤섞여 있어서, **그대로는 세지도 비교하지도 못합니다**.

그래서 글을 분석하기 전에 반드시 거치는 관문이 **텍스트 전처리**입니다 — 지저분한 문자를 걷어내고(**정제**), 문장을 의미 단위로 쪼개고(**토큰화**), 한국어 특유의 조사·어미를 떼어내고(**형태소 분석**), 의미 없는 단어를 버립니다(**불용어 제거**). 이번 시간엔 이 네 단계를 하나의 **전처리 파이프라인 함수**로 완성합니다. 이 함수가 다음 시간과 과제에서 계속 쓰입니다.

## ⏪ 복습 — 지난 시간: 표 데이터와 시각화

지난 시간까지 우리는 **정형 데이터**를 다뤘습니다.

- **pandas**: `read_csv` 로 표를 불러와 `head()`·`info()`·`describe()` 로 살펴보고, 열을 골라 값을 셌습니다.
- **matplotlib·seaborn**: 숫자와 범주의 분포·비교를 **그림으로** 확인했습니다.
- **`collections.Counter`**: 값이 몇 번 나왔는지 세는 도구도 이미 배웠습니다 — 이번 시간의 **단어 세기**에 그대로 씁니다.

표에서 `age` 열은 이미 숫자라 바로 평균을 낼 수 있었습니다. 하지만 **리뷰 한 줄**은 무엇을 세야 할까요? 글자? 어절? 단어? — **무엇을 셀지 정하는 일**이 바로 이번 시간의 전처리입니다.

**오늘의 목표**

- [ ] 원시 텍스트에 어떤 **노이즈**가 있는지 눈으로 찾아낸다.
- [ ] 문자열의 `find`와 정규표현식의 `search`·`findall`을 목적에 맞게 골라 **찾는다**.
- [ ] **정규표현식**(`re.sub`)으로 특수문자는 **정제**하고, 반복문자·공백은 같은 기준으로 **정규화**한다.
- [ ] **토큰화**가 무엇인지, 한국어에서 공백 `split()` 이 왜 부족한지 설명한다.
- [ ] **한국어 형태소 분석기(kiwipiepy)** 로 품사를 보고 명사·형용사·동사만 골라낸다.
- [ ] **일반 불용어**를 적용하고, 빈도를 보고 **도메인 불용어**를 직접 만들어 추가한다.
- [ ] 위 단계를 **전처리 파이프라인 함수**(`clean_text`·`tokenize`)로 완성한다.

아래 셀을 먼저 실행해 라이브러리와 한국어 형태소 분석기를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

## 데이터 살펴보기 — 영화 리뷰 2000건

이번 시간에 다룰 데이터는 실제 관객이 남긴 **영화 리뷰** 2000건입니다. `text`(리뷰 본문)와 `label`(1=긍정, 0=부정) 두 열로 되어 있습니다.

새 데이터를 만나면 늘 하던 대로 **생김새부터** 봅니다 — 앞부분(`head`)·구조와 결측(`info`)·분포(`value_counts`).

In [ ]:
reviews = pd.read_csv('data/movie_reviews.csv')

print('리뷰 크기:', reviews.shape)
print('\n[앞부분] head()')
display(reviews.head())
print('\n[구조와 결측] info()')
reviews.info()
print('\n[감성(label) 분포]  1=긍정, 0=부정')
display(reviews['label'].value_counts().to_frame('건수'))

### 이대로는 못 셉니다 — 원문의 노이즈를 눈으로

표 데이터라면 여기서 바로 평균을 냈겠지만, 텍스트는 다릅니다. **원문을 몇 개만 그대로 찍어 보세요.**

오늘 우리가 할 일을 먼저 한 장으로 봅시다. 리뷰 한 문장이 **셀 수 있는 단어 목록**이 되기까지 네 단계를 거칩니다. 지금부터 이 단계를 하나씩 손으로 만들어 갑니다.

<img src="images/교안/전처리_파이프라인.png" width="900" style="max-width:100%"/>

In [ ]:
# 원문을 있는 그대로 몇 개 출력 — 무엇이 문제인지 직접 찾아보자
for i in [0, 3, 8, 10]:
    print(f"[{i}] {reviews.loc[i, 'text']}")

print('\n무엇이 보이나요?')
print('  - 특수문자·문장부호: ! ? . , ~ 가 뜻 없이 붙어 있다')
print('  - 반복 문자: ㅋㅋㅋ, ...., !!!!!! 처럼 같은 글자가 늘어진다')
print('  - 붙여 쓴 오타: 개재미없다, 너무재밌어서')
print('  - 한국어 조사: 영화가 / 영화는 / 영화를 — 사람 눈엔 같은 단어지만 컴퓨터엔 전부 다른 글자')

컴퓨터에게 `영화가`·`영화는`·`영화를`은 **전혀 다른 세 단어**입니다. `최고!!!` 와 `최고` 도 다른 단어입니다. 이 상태로 단어를 세면 같은 뜻이 산산조각 나서 **아무것도 안 보입니다**. 그래서 세기 전에 **정제 → 토큰화 → 형태소 → 불용어 제거** 순서로 텍스트를 다듬습니다.

---
# 1. 정제·정규화 — 정규표현식으로 노이즈를 걷고 표기를 맞추기

## 왜 필요할까요?
리뷰에 붙은 `!!!`·`....`·이모지·`ㅋㅋㅋㅋ` 는 **뜻을 거의 담지 않으면서 단어를 오염**시킵니다. 이런 잡음을 규칙으로 걸러 내는 일을 **정제(cleaning)** 라 하고, 그 규칙을 적는 문법이 **정규표현식**입니다.

## 비유
정규표현식은 **글자판 위의 그물**입니다. "한글도 영어도 숫자도 공백도 **아닌** 글자를 모두 건져 내라" 처럼 **패턴**을 말하면, 파이썬이 문장 전체를 훑으며 걸리는 것을 잡아 줍니다. 글자 하나하나를 손으로 지우는 대신 **규칙 한 줄**로 끝냅니다.

### 패턴 문법 — 작은 부품을 조립한다
| 표기 | 뜻 | 예 |
|---|---|---|
| `abc` | 글자 그대로 | `영화` = 정확히 '영화' |
| `[...]` | 대괄호 안 글자 중 **하나** | `[가-힣]` = 한글 한 글자 |
| `[^...]` | `^` 는 **부정** — 대괄호 안의 것이 **아닌** 글자 | `[^가-힣]` = 한글이 아닌 글자 |
| `\d` · `\s` · `\S` | 숫자 · 공백 · 공백 아닌 글자 | `\d+` = 이어진 숫자 |
| `.` | 줄바꿈을 제외한 아무 글자 하나 | `가.` = 가 뒤에 글자 하나 |
| `*` · `+` · `?` | 앞 패턴이 0회 이상 · 1회 이상 · 0/1회 | `https?` = http 또는 https |
| `{m,n}` | 앞 패턴이 m~n회 | `\d{2,3}` = 숫자 2~3개 |
| `^` · `$` | 문자열의 시작 · 끝 | `^공지` = 공지로 시작 |
| `(...)` | 여러 글자를 한 덩어리로 묶고 기억 | `(좋아)+` |
| `A\|B` | A 또는 B | `긍정\|부정` |
| `(.)` 와 `\1` | 괄호로 **묶어 기억**했다가 `\1` 로 **다시 부른다**(역참조) | `(.)\1{2,}` = 같은 글자가 3번 이상 |

> 패턴 문자열 앞에는 **`r`** 을 붙입니다(`r'\s+'`). 역슬래시를 파이썬이 먼저 해석하지 않게 하는 표시입니다.

### 찾기·검사·바꾸기 — 함수는 목적에 맞게 고른다
| 목적 | 함수 | 결과 |
|---|---|---|
| 정확한 글자 위치 찾기 | `문장.find('글자')` | 시작 위치, 없으면 `-1` — **정규표현식 아님** |
| 패턴의 첫 일치 찾기 | `re.search(패턴, 문장)` | `Match` 또는 `None` |
| 모든 일치값 뽑기 | `re.findall(패턴, 문장)` | 문자열 리스트 |
| 모든 일치 부분 바꾸기 | `re.sub(패턴, 바꿀값, 문장)` | 바뀐 문자열 |

> 정규표현식 모듈에는 `re.find()`가 없습니다. 글자 그대로 찾으면 문자열의 `find`, 패턴을 찾으면 `re.search`·`re.findall`을 사용합니다.

> <strong>선택 기준:</strong> 글자 그대로 찾으면 `find`, 패턴의 첫 한 건이면 `search`, 값만 전부 모으면 `findall`, 바꾸려면 `sub`입니다.
`find`의 반환값은 시작 위치가 0이면 거짓, 못 찾은 `-1`은 참처럼 평가될 수 있으므로 `if text.find(...)`로 존재 여부를 검사하지 말고 `'글자' in text` 또는 `text.find(...) != -1`을 사용합니다.

### 우리가 쓸 정제 3단계
1. **특수문자 제거** — `[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]` 에 걸리는 글자를 공백으로 (한글 음절과 `ㅋㅋ` 같은 자모는 남김)
2. **반복문자 축약** — `(.)\1{2,}` → `\1\1` (같은 글자 3번 이상이면 2번으로: `ㅋㅋㅋㅋ` → `ㅋㅋ`)
3. **공백 정규화** — `\s+` → 한 칸, 그리고 양끝 공백 제거

> **정제와 정규화는 다릅니다.** 정제는 분석에 필요 없는 특수문자·이모지를 **없애는 일**이고, 정규화는 `ㅋㅋㅋㅋ`와 `ㅋㅋㅋ`, 여러 칸 공백처럼 같은 뜻의 표기 변형을 **하나의 기준으로 맞추는 일**입니다. 뒤의 형태소 분석도 `영화가`·`영화를`을 `영화`로 모아 형태 변이를 줄인다는 점에서 정규화 역할을 합니다.

### `find`와 `search` — 글자 그대로인가, 패턴인가

`str.find`는 정확한 글자만 찾고, `re.search`는 정규표현식 패턴의 **첫 일치**를 찾습니다. `search` 결과는 문자열이 아니라 `Match` 객체이므로, 찾았는지 먼저 확인한 뒤 `.group()`으로 값을 꺼냅니다.

In [ ]:
# 정확한 글자 위치와 정규표현식의 첫 일치를 비교한다
message = '문의번호 CS-102, 주문번호 OD-305, 연락처 010-1234-5678'

literal_index = message.find('주문번호')
first_code = re.search(r'[A-Z]{2}-\d{3}', message)

print('find 결과:', literal_index)
if first_code is not None:
    print('search 값 :', first_code.group())

### `findall` — 일치하는 값을 모두 모으기

`findall`은 패턴에 맞는 값을 바로 리스트로 주어 집계에 편합니다. 첫 값만 필요한 `search`와 달리 문장 전체의 모든 값을 모읍니다.

In [ ]:
# 같은 패턴에 맞는 모든 값을 리스트로 모은다
code_pattern = r'[A-Z]{2}-\d{3}'
all_codes = re.findall(code_pattern, message)
print('findall:', all_codes)

### `sub` — 일치하는 부분을 모두 바꾸기

`re.sub(패턴, 바꿀값, 문장)`은 문장에서 패턴에 맞는 부분을 **모두 찾아 바꾼 새 문자열**을 반환합니다. 원본 문자열 자체는 바뀌지 않으므로 결과를 새 변수에 담아야 합니다.

| 인자 | 역할 | 예 |
|---|---|---|
| `패턴` | 무엇을 찾을지 정함 | `r'\d+'` = 이어진 숫자 |
| `바꿀값` | 찾은 부분을 무엇으로 바꿀지 정함 | `'[숫자]'` 또는 `' '` |
| `문장` | 찾고 바꿀 원본 문자열 | `'주문 123, 문의 456'` |

```python
re.sub(r'\d+', '[숫자]', '주문 123, 문의 456')
# 결과: '주문 [숫자], 문의 [숫자]'
```

텍스트를 정제할 때는 특수문자를 빈 문자열 `''`보다 **공백 `' '`으로 바꾸는 편이 안전**합니다. 예를 들어 `좋다!!!최고`에서 `!!!`을 빈 문자열로 지우면 `좋다최고`로 붙지만, 공백으로 바꾸면 `좋다 최고`처럼 단어 경계가 남습니다. 반복문자처럼 앞에서 기억한 글자를 다시 쓸 때는 바꿀값에 역참조를 넣습니다: `re.sub(r'(.)\1{2,}', r'\1\1', 문장)`.

In [ ]:
# 정제 3단계를 한 문장에 차례로 적용해 보자
sample = 'TV시리즈가 너무재밌어서   영화는 기대안하고 봤는데 역시....최고네요ㅋㅋㅋㅋ!!!'
print('원문   :', sample)

# 1) 특수문자·이모지 제거 — 한글·영문·숫자·공백이 '아닌' 글자를 공백으로
step1 = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', sample)
print('1단계  :', step1)

# 2) 반복문자 축약 — 같은 글자가 3번 이상이면 2번으로 (ㅋㅋㅋㅋ → ㅋㅋ)
step2 = re.sub(r'(.)\1{2,}', r'\1\1', step1)
print('2단계  :', step2)

# 3) 공백 정규화 — 여러 칸을 한 칸으로, 양끝 공백 제거
step3 = re.sub(r'\s+', ' ', step2).strip()
print('3단계  :', step3)

특수문자가 사라지고 `ㅋㅋㅋㅋ` 가 `ㅋㅋ` 로 줄었으며, 늘어진 공백도 한 칸으로 정리됐습니다. 이제 이 세 줄을 **함수 하나**로 묶으면 어떤 리뷰에도 똑같이 적용할 수 있습니다(마지막 응용에서 완성합니다).

### 실무에서 자주 쓰는 정제 패턴
우리 리뷰 데이터는 비교적 깨끗하지만, **웹에서 수집한 텍스트**(지난 단원에서 해 봤죠)에는 HTML 태그·링크·이메일·전화번호가 그대로 섞여 들어옵니다. 이런 것들은 **내용이 아니라 껍데기**라 가장 먼저 걷어 냅니다. 아래는 **원리를 익히기 위한 단순 패턴**입니다. 실제 HTML은 중첩·속성·깨진 태그가 있어 BeautifulSoup 같은 HTML 파서를 쓰는 편이 안전하고, 숫자·이모지는 분석 목적에 따라 중요한 신호일 수 있으므로 무조건 지우지 말고 남길지 먼저 판단합니다.

| 무엇을 지우나 | 패턴 | 설명 |
|---|---|---|
| HTML 태그 | `<[^>]+>` | `<` 로 시작해 `>` 가 아닌 글자가 이어지다 `>` 로 끝나는 덩어리 |
| 링크(URL) | `https?://\S+` | `http` 또는 `https`(`s?` = s 가 있어도 없어도) 뒤에 공백 아닌 글자들 |
| 이메일 | `\S+@\S+\.\S+` | 공백 아닌 글자 + `@` + 공백 아닌 글자 + `.` + 공백 아닌 글자 |
| 숫자 | `\d+` | 숫자 1개 이상 (전화번호·가격처럼 분석에 방해되면 제거) |

> `\S` 는 **공백이 아닌 글자**, `\d` 는 **숫자**입니다. `?` 는 **앞의 것이 0번 또는 1번**이라는 뜻이라 `https?` 는 `http` 와 `https` 를 한꺼번에 잡습니다.

In [ ]:
# 웹에서 긁어온 듯한 지저분한 문장을 실무 패턴으로 정제해 보자
raw = '<p>배송 문의는 <b>help@shop.co.kr</b> 로!</p> 자세한 내용 https://shop.co.kr/faq 참고. 전화 01012345678'
print('원문      :', raw)

# 1) HTML 태그 제거
t = re.sub(r'<[^>]+>', ' ', raw)
print('태그 제거 :', t.strip())

# 2) 링크(URL) 제거
t = re.sub(r'https?://\S+', ' ', t)
print('링크 제거 :', t.strip())

# 3) 이메일 제거
t = re.sub(r'\S+@\S+\.\S+', ' ', t)
print('메일 제거 :', t.strip())

# 4) 숫자 제거 + 공백 정리
t = re.sub(r'\d+', ' ', t)
t = re.sub(r'\s+', ' ', t).strip()
print('최종      :', t)

껍데기(태그·링크·메일·번호)가 사라지고 **사람이 쓴 말만** 남았습니다. 실무에서는 이 패턴들을 **가장 먼저** 적용한 뒤, 앞의 정제 3단계로 넘어갑니다. (이번 시간의 리뷰 데이터에는 태그·링크가 없어 3단계만으로 충분합니다.)

### 🖐️ 함께 따라하기 — 고객 문의에서 필요한 패턴 찾기

앞의 고객 문의와 다른 **시스템 로그**에서 정확한 글자 위치, 첫 코드와 모든 코드를 찾아봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 0) log 문자열 변수에 'ERROR 코드 ER-104 발생, 재시도 코드 RT-208 기록' 을 담는다
#    code_pattern 문자열 변수에는 영문 대문자 2개-숫자 3개를 찾는 정규표현식을 담는다
# 1) log 변수에서 정확한 글자 '재시도'의 시작 위치를 str.find 로 찾아 retry_index 변수에 담는다
#    retry_index 는 정수이며, 찾지 못하면 -1 이다
# 2) re.search(code_pattern, log) 결과를 first_match 변수에 담는다
#    일치하면 first_match.group() 문자열을 first_code 변수에, 없으면 None 을 담는다
# 3) re.findall(code_pattern, log)로 모든 일치 문자열을 찾아 all_codes 리스트 변수에 담는다
# 4) retry_index, first_code, all_codes 변수를 각각 출력한다

### ✅ 바로 확인 퀴즈

**1.** `re.sub(r'[^가-힣]', '', '최고!!! good 100점')` 의 결과는 무엇일까요?

<details><summary>정답 보기</summary>

`'최고점'` 입니다. `[^가-힣]` 은 **한글이 아닌 모든 글자**(느낌표·공백·영문·숫자)를 뜻하고, 그것을 빈 문자열 `''` 로 바꿨으니 한글만 남습니다.

</details>

**2.** 패턴 `(.)\1{2,}` 는 어떤 문자열에 걸릴까요? `ㅋㅋ` 와 `ㅋㅋㅋ` 중 걸리는 것은?

<details><summary>정답 보기</summary>

**`ㅋㅋㅋ`** 입니다. `(.)` 가 글자 하나를 기억하고 `\1{2,}` 가 **그 글자가 2번 더** 나오길 요구하므로 **같은 글자 3번 이상**이어야 걸립니다. `ㅋㅋ`(2번)는 걸리지 않습니다.

</details>

**3.** 패턴에 맞는 첫 한 건만 찾을 때와 모든 값을 리스트로 모을 때 각각 어떤 함수를 쓰나요?

<details><summary>정답 보기</summary>

첫 한 건은 `re.search`, 모든 값을 리스트로 모을 때는 `re.findall`을 씁니다.

</details>

**4.** `문장.find('010')` 과 `re.search(r'\d{3}-\d{4}-\d{4}', 문장)`의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

`find`는 정확한 글자 `010`의 시작 위치를 숫자로 돌려주며 정규표현식을 해석하지 않습니다. `search`는 전화번호처럼 바뀔 수 있는 **패턴**의 첫 일치를 `Match` 객체로 돌려줍니다.

</details>

---
# 2. 토큰화 — 문장을 단어로 쪼개기

## 왜 필요할까요?
정제한 문장은 여전히 **한 덩어리 문자열**입니다. 단어를 세려면 문장을 **의미 단위(토큰)** 로 쪼개야 합니다. 이 작업이 **토큰화(tokenization)** 입니다.

## 비유
레고 작품을 **블록 단위로 분해**하는 일입니다. 분해해야 어떤 블록이 몇 개 쓰였는지 셀 수 있습니다.

### 가장 단순한 방법: 공백으로 자르기
- **`문장.split()`** → 공백을 기준으로 쪼갠 리스트. 영어라면 이걸로도 제법 쓸 만합니다.

### 여러 문장의 단어를 누적해서 셀 때: `Counter`
- **`Counter()`** → `{단어: 등장 횟수}`를 기억하는 빈 계수 객체.
- **`.update(토큰리스트)`** → 리스트의 각 단어를 현재 횟수에 누적.
- **`.most_common(k)`** → 빈도가 높은 `(단어, 횟수)` 튜플을 k개 담은 리스트.

> 여기서 세는 값은 **그 단어가 나온 문서 수**가 아니라, 전체 리뷰에 등장한 **토큰 횟수**입니다.

### 그런데 한국어에서는 왜 부족할까요?
한국어는 명사 뒤에 **조사**가 달라붙습니다 — `영화가`·`영화는`·`영화를`·`영화도`. 공백으로만 자르면 이들이 **전부 다른 단어**로 세어집니다. 동사·형용사도 어미가 바뀝니다 — `재밌어요`·`재밌다`·`재밌네`. **뜻은 하나인데 형태가 수십 가지**가 되는 것이죠.

말로만 들으면 와닿지 않으니, 실제 리뷰 2000건에서 `영화`로 시작하는 형태를 세어 본 결과를 보세요.

<img src="images/교안/공백분리_vs_형태소.png" width="880" style="max-width:100%"/>

In [ ]:
# 공백 split 으로 쪼개 보면 — 같은 단어가 조사 때문에 갈라진다
sent = '이 영화가 정말 재밌었어요 이 영화는 최고 영화를 또 보고 싶다'
print('공백 토큰:', sent.split())

# 전체 리뷰를 공백으로만 쪼개 상위 단어를 세어 보자
split_counter = Counter()
for t in reviews['text']:
    split_counter.update(t.split())
display(split_counter)

In [ ]:
print('\n[공백 split 으로 센 상위 10개]')
for word, n in split_counter.most_common(10):
    print(f'  {word} : {n}')
print('\n같은 뜻인 영화가/영화는/영화를 이 따로따로 세어지고, 문장부호까지 단어에 붙어 있다')

### 🖐️ 함께 따라하기 — 배송 문의에서 같은 말이 몇 가지로 갈라졌나

영화 리뷰 시연과 다른 **고객센터 배송 문의**를 공백으로 잘라 봅니다. `support_counter.items()`는 `(단어, 빈도)` 쌍을 꺼내고, `단어.startswith('배송')`는 해당 단어가 `'배송'`으로 시작하면 `True`를 반환합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 0) support_messages 리스트 변수에 제공된 배송 문의 문자열 5개를 담는다
# 1) 빈 Counter 객체를 support_counter 변수에 만들고, 각 문의를 split한 결과를 누적한다
# 2) support_counter.items()에서 '배송'으로 시작하는 (단어, 빈도) 튜플만 골라
#    delivery_forms 리스트 변수에 담는다(startswith 사용)
# 3) delivery_forms 리스트를 각 튜플의 두 번째 값(빈도) 기준 내림차순으로 정렬한다
# 4) 서로 다른 형태 수와 delivery_forms 리스트의 모든 단어·횟수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `'나는 이 영화가 좋다'.split()` 의 결과는 몇 개의 토큰인가요?

<details><summary>정답 보기</summary>

**4개** 입니다 — `['나는', '이', '영화가', '좋다']`. 공백을 기준으로만 자르기 때문입니다.

</details>

**2.** 한국어에서 공백 `split()` 만으로 토큰화하면 생기는 가장 큰 문제는 무엇인가요?

<details><summary>정답 보기</summary>

**조사·어미가 붙은 채로 잘려서 같은 단어가 여러 형태로 갈라집니다.** `영화가`·`영화는`·`영화를` 이 각각 다른 단어로 세어지므로, 정작 `영화` 라는 단어의 진짜 빈도를 알 수 없습니다.

</details>

---
# 3. 한국어 형태소 분석 — 조사를 떼고 품사를 본다

## 왜 필요할까요?
`영화가`에서 **`영화`(명사)** 와 **`가`(조사)** 를 갈라내면, 조사가 뭐든 `영화` 하나로 셀 수 있습니다. 이렇게 **뜻을 가진 최소 단위(형태소)** 로 쪼개고 각 조각의 **품사**까지 알려 주는 도구가 **형태소 분석기**입니다. 우리는 설치가 간단한 **kiwipiepy** 를 씁니다(자바가 필요 없습니다).

## 비유
`재밌었어요` 를 `재밌`(형용사) + `었`(과거) + `어요`(종결어미) 로 **분해**하는 일입니다. 우리가 세고 싶은 건 **`재밌`** 이지 어미가 아닙니다.

### 문법
- **`kiwi.tokenize(문장)`** → 토큰 객체들의 리스트.
- 토큰 하나에서 꺼낼 것 두 가지: **`.form`**(글자 모양) · **`.tag`**(품사 태그).

### 품사 태그 — 이번 시간에 쓸 것만
| 태그 | 품사 | 예 |
|---|---|---|
| `NNG`·`NNP` (**`NN`** 으로 시작) | 일반명사·고유명사 | 영화, 배우, 스토리 |
| **`VA`** | 형용사 | 재밌, 좋, 아름답 |
| **`VV`** | 동사 | 보, 만들, 웃 |
| `JKS`·`JX` (조사) | 조사 | 가, 는, 를 |
| `EP`·`EF` (어미) | 어미 | 었, 어요 |

감성을 담는 말은 대부분 **명사·형용사·동사**입니다. 그래서 태그가 **`NN`·`VA`·`VV`** 로 시작하는 토큰만 남기고, **한 글자짜리**(`것`·`수`·`이`처럼 뜻이 약함)는 버립니다.

In [ ]:
# 형태소 분석기가 문장을 어떻게 쪼개는지 (글자 모양 .form / 품사 .tag)
sent = '이 영화가 정말 재밌었어요'
print('공백 split :', sent.split())
print('\n[형태소 분석 결과]')
for t in kiwi.tokenize(sent):
    print(f'  {t.form:6s} {t.tag}')

# 명사(NN)·형용사(VA)·동사(VV) 이면서 2글자 이상인 것만 남긴다
picked = [t.form for t in kiwi.tokenize(sent)
          if t.tag.startswith(('NN', 'VA', 'VV')) and len(t.form) > 1]
print('\n골라낸 토큰:', picked)
print('조사(가)·부사(정말)·어미(었/어요)가 사라지고 뜻을 가진 말만 남았다')

`영화가` → **`영화`**, `재밌었어요` → **`재밌`**. 이제 조사·어미가 달라도 **같은 단어로 세어집니다**. 정제(1절)와 형태소 분석(3절)을 이어 붙이면 지저분한 원문에서도 깔끔한 단어 목록이 나옵니다.

> **이번 필터는 키워드 탐색용 단순 규칙입니다.** `NN`·`VA`·`VV`와 2글자 이상만 남기면 `안 좋다`의 `안`, `재밌지 않다`의 `않`, `돈 아깝다`의 `돈`처럼 감성 방향이나 핵심을 바꾸는 1글자·보조 표현이 빠질 수 있습니다. 감성 방향을 판단할 때는 제거된 부정 표현과 1글자 핵심어를 원문에서 함께 점검합니다.

In [ ]:
# 단순 품사·길이 필터가 놓칠 수 있는 부정 표현과 1글자 핵심어를 확인한다
for sample in ['재밌지 않다', '안 좋다', '돈 아깝다']:
    kept = [token.form for token in kiwi.tokenize(sample)
            if token.tag.startswith(('NN', 'VA', 'VV')) and len(token.form) > 1]
    print(f'{sample:10s} 단순 필터={kept}')

### 🖐️ 함께 따라하기 — 고객 문의 한 건 정제·형태소 분석하기

영화 리뷰 시연과 다른 **고객센터 문의 한 건**을 정제한 뒤 형태소 분석해, 명사·형용사·동사(2글자 이상)만 뽑아 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '배송이 너무느리고 상담원이 답답했어요!!!' 문자열을 raw 변수에 담아 출력한다
# 2) raw 변수에 정제 3단계(특수문자 제거 → 반복문자 축약 → 공백 정규화)를 적용한다
#    정제된 문자열을 cleaned 변수에 담는다
# 3) kiwi.tokenize(cleaned) 결과에서 태그가 NN·VA·VV 로 시작하고 2글자 이상인 form 만 고른다
#    고른 문자열들을 tokens 리스트 변수에 담는다
# 4) cleaned 문자열 변수와 tokens 리스트 변수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 토큰 하나에서 **글자 모양**과 **품사**를 꺼내려면 각각 무엇을 쓰나요?

<details><summary>정답 보기</summary>

글자 모양은 **`.form`**, 품사는 **`.tag`** 입니다. 예: `t.form` → `'영화'`, `t.tag` → `'NNG'`.

</details>

**2.** 태그가 `NN` 으로 시작하는 토큰만 남기면 무엇이 남나요? 왜 형용사(`VA`)·동사(`VV`)도 함께 남길까요?

<details><summary>정답 보기</summary>

`NN` 만 남기면 **명사**만 남습니다. 하지만 리뷰의 감정은 `재밌`·`아깝`·`좋` 같은 **형용사**와 `만들`·`보` 같은 **동사**에 많이 담겨 있어서, 감성 분석에서는 `VA`·`VV` 도 함께 남깁니다.

</details>

---
# 4. 불용어 제거 — 일반 불용어, 그리고 도메인 불용어

## 왜 필요할까요?
형태소로 잘 쪼갰어도 **뜻이 거의 없는 흔한 말**은 여전히 남습니다. 이런 단어를 **불용어(stopword)** 라 하고, 세기 전에 버립니다. 불용어는 **두 종류**입니다.

### ① 일반 불용어 — 어느 글에나 흔한 말
`것`·`수`·`등`·`그리고` 처럼 **어떤 주제의 글이든** 무의미하게 자주 나오는 말. 이런 목록은 **직접 만들 필요 없이 공개된 것을 받아** 씁니다. 우리가 쓰는 `data/stopwords_ko.json`(679개)도 아래 목록을 미리 받아 둔 것입니다.

> **받는 곳** — `github.com/stopwords-iso/stopwords-ko` (txt) · `github.com/6/stopwords-json` (json, 내용 동일)

```
# 인터넷에서 바로 받아 쓰기 (json 판)
import json, urllib.request
url = 'https://raw.githubusercontent.com/6/stopwords-json/master/dist/ko.json'
stopwords = set(json.load(urllib.request.urlopen(url)))   # 679개
```

### 로컬 JSON 불용어 파일을 읽는 핵심 문법
- **`with open(경로, encoding='utf-8') as f`** → 파일을 열고 블록이 끝나면 자동으로 닫습니다.
- **`json.load(f)`** → JSON 배열을 파이썬 리스트로 읽습니다.
- **`set(...)`** → 리스트를 집합으로 바꿉니다. 중복을 없애고 `단어 in stopwords` 검사를 빠르게 합니다.
- **`A | B`** → 두 집합을 합친 새 집합을 만듭니다. 뒤에서 일반 불용어와 도메인 불용어를 합칠 때 씁니다.

### ② 도메인 불용어 — **이 데이터에서만** 무의미한 말
영화 리뷰에서 `영화` 라는 단어가 자주 나오는 건 **당연**합니다. 전부 영화 이야기니까요. 그래서 `영화` 는 이 데이터에서 **아무것도 구분해 주지 못하는 말**입니다. `평점` 도 마찬가지죠. 하지만 뉴스 기사에서 `영화` 가 나오면 그건 **의미 있는 단어**입니다.

> **도메인 불용어는 목록을 미리 알 수 없습니다. 데이터를 봐야 압니다.** 먼저 일반 불용어만 적용해 **빈도 상위를 눈으로 확인**하고, 그중 "이 도메인에서만 무의미한 고빈도어"를 골라 **직접 목록을 만드는 것** — 이게 실무 전처리의 핵심 작업입니다.

In [ ]:
# 일반 불용어 679개를 불러온다
with open('data/stopwords_ko.json', encoding='utf-8') as f:
    stopwords_general = set(json.load(f))

print('일반 불용어 개수:', len(stopwords_general))
print('예시:', sorted(stopwords_general)[:15])

In [ ]:
# 정제 + 형태소 + 일반 불용어까지 적용해 전체 리뷰의 단어 빈도를 세어 본다
def tokenize_basic(text, stopwords):
    """정제 → 형태소 → 품사 필터 → 불용어 제거 (아직 도메인 불용어는 없다)."""
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', str(text))
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return [t.form for t in kiwi.tokenize(text)
            if t.tag.startswith(('NN', 'VA', 'VV'))
            and len(t.form) > 1 and t.form not in stopwords]

counter_general = Counter()
for t in reviews['text']:
    counter_general.update(tokenize_basic(t, stopwords_general))

print('[일반 불용어만 적용한 빈도 top15]')
for word, n in counter_general.most_common(15):
    print(f'  {word:5s} {n}')

**빈도표를 눈으로 읽어 봅시다.** `영화`(930회)가 압도적 1위이고 `평점`(140회)이 2위입니다. 그 아래로 `생각`·`사람` 도 보입니다.

- `영화`·`평점` — **영화 리뷰 데이터니까 당연히 많은 말**입니다. 긍정 리뷰에도, 부정 리뷰에도 똑같이 나옵니다. **아무것도 구분해 주지 못합니다.**
- `생각`·`사람`·`정말`·`진짜`·`그냥` — 리뷰라면 어디에나 붙는 **말버릇**입니다.
- 반면 `재밌`·`감동`·`연기`·`스토리` 는 **남겨야 합니다.** 이건 영화 이야기의 **알맹이**입니다.

→ 앞의 것들만 골라 **도메인 불용어 목록**을 만듭니다. 목록은 빈도표를 보고 **사람이 판단**해 정합니다.

In [ ]:
# 빈도표를 보고 '이 도메인에서만 무의미한 고빈도어'를 직접 골라 도메인 불용어를 만든다
stopwords_domain = {'영화', '평점', '배우'}
stopwords_all = stopwords_general | stopwords_domain   # 일반 + 도메인
print('전체 불용어 개수:', len(stopwords_all))

# 도메인 불용어까지 적용해 다시 센다
counter_all = Counter()
for t in reviews['text']:
    counter_all.update(tokenize_basic(t, stopwords_all))

# 적용 전후 top10 을 나란히 비교
before = [f'{w} ({n})' for w, n in counter_general.most_common(10)]
after = [f'{w} ({n})' for w, n in counter_all.most_common(10)]
compare = pd.DataFrame({'일반 불용어만': before, '일반 + 도메인 불용어': after},
                       index=range(1, 11))
compare.index.name = '순위'
display(compare)
print('영화·평점·생각·사람이 빠지고, 그 자리를 감동·최고·배우 같은 알맹이 단어가 채웠다')

왼쪽 열의 1·2위였던 `영화`·`평점`이 사라지자, 오른쪽 열에는 **`재밌`·`연기`·`드라마`·`스토리`·`감동`·`최고`·`배우`** 처럼 리뷰의 내용을 실제로 말해 주는 단어만 남았습니다. 같은 데이터인데 **볼 수 있는 것이 달라졌습니다.**

**기억할 것** — 도메인 불용어 목록은 데이터마다 **다시 만들어야 합니다**. 제품 리뷰라면 `제품`·`구매`·`배송` 이, 뉴스 헤드라인이라면 `종합`·`속보` 가 그 자리에 옵니다. **빈도 상위를 보고 고르는 습관**이 곧 실력입니다.

### 🖐️ 함께 따라하기 — 고객 문의의 도메인 불용어 만들기

영화 리뷰 시연과 다른 **고객센터 문의 6건**의 빈도 상위를 보고, 모든 문의에 반복돼 분류에 도움이 적은 도메인 단어를 직접 골라 제거합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 0) support_texts 리스트 변수에 제공된 고객 문의 문자열 6개를 담는다
# 1) 일반 불용어만 적용한 토큰을 support_counter Counter 변수에 누적해 top10을 출력한다
# 2) 고객 문의 분류에 도움 적은 '문의'와 '상담'을 support_domain_stopwords 집합 변수에 담는다
# 3) stopwords_general | support_domain_stopwords 결과를 support_stopwords 집합 변수에 담는다
# 4) 같은 문의들을 support_stopwords로 다시 토큰화해 support_counter_v2 Counter 변수에 누적한다
# 5) support_counter_v2.most_common(10)을 출력해 두 단어가 사라졌는지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 영화 리뷰에서 `영화` 는 도메인 불용어입니다. 그렇다면 **뉴스 기사** 데이터에서도 `영화` 를 버려야 할까요?

<details><summary>정답 보기</summary>

아니요. 뉴스에서 `영화` 는 **문화 기사를 가려내는 의미 있는 단어**입니다. 도메인 불용어는 **그 데이터에서만** 무의미한 말이므로, 데이터가 바뀌면 목록도 다시 만들어야 합니다.

</details>

**2.** 도메인 불용어 목록은 어떻게 정하나요?

<details><summary>정답 보기</summary>

**데이터를 봐야 압니다.** 먼저 일반 불용어만 적용해 **빈도 상위 단어를 확인**하고, 그중 "모든 문서에 똑같이 나와서 아무것도 구분해 주지 못하는 말"을 사람이 골라 목록으로 만듭니다.

</details>

**3.** 일반 불용어 목록만 쓰고 도메인 불용어를 만들지 않으면 어떤 일이 벌어지나요?

<details><summary>정답 보기</summary>

`영화`(930회)처럼 **압도적이지만 아무 정보도 없는 단어**가 빈도표 1위를 차지해, 정작 봐야 할 `재밌`·`감동`·`스토리` 같은 알맹이 단어가 묻힙니다.

</details>

---
## 🚀 응용 클론코딩 — 전처리 파이프라인 함수 완성

지금까지 배운 네 단계를 **재사용 가능한 함수 두 개**로 묶습니다. **이 두 함수가 다음 시간과 모든 과제에서 계속 쓰이는 표준 도구**가 됩니다.

| 함수 | 하는 일 |
|---|---|
| `clean_text(text)` | 정제 3단계 — 특수문자 제거 → 반복문자 축약 → 공백 정규화 (문자열 반환) |
| `tokenize(text, stopwords)` | `clean_text` 를 거친 뒤 형태소 분석 → 품사 필터(`NN`·`VA`·`VV`) → 2글자 이상 → 불용어 제거 (리스트 반환) |

완성한 뒤 시연 데이터와 다른 고객 문의 몇 건에 적용해 원문과 토큰을 나란히 확인해 보세요.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) clean_text(text) 함수를 정의한다
#    1-1. 한글·영문·숫자·공백이 아닌 글자를 공백으로 바꾼다
#    1-2. 같은 글자가 3번 이상 반복되면 2번으로 줄인다(역참조 사용)
#    1-3. 여러 공백을 한 칸으로 줄이고 양끝 공백을 제거해 돌려준다
# 2) tokenize(text, stopwords) 함수를 정의한다
#    2-1. clean_text 를 거친 문장을 kiwi.tokenize 로 분석한다
#    2-2. 태그가 NN·VA·VV 로 시작하고, 2글자 이상이고, 불용어가 아닌 토큰의 form 만 리스트로 돌려준다
# 3) support_samples 리스트 변수에 고객 문의 문자열 3개를 담고,
#    각 문자열에 tokenize(text, support_stopwords)를 적용해 원문과 토큰을 함께 출력한다

---
## 이번 강의 정리

| 단계 | 하는 일 | 도구 |
|---|---|---|
| 패턴 찾기 | 정확한 글자·첫 패턴·모든 패턴 찾기 | `str.find`, `re.search`, `re.findall` |
| 정제 | 분석에 필요 없는 특수문자·이모지 제거 | `re.sub` |
| 정규화 | 반복문자·공백·형태 변이를 같은 기준으로 맞춤 | `re.sub` · 형태소 분석 |
| 토큰화 | 문장을 단어 단위로 쪼갬 | `split()`(영어) → 한국어엔 부족 |
| 형태소 분석 | 조사·어미를 떼고 품사를 봄 | `kiwi.tokenize` → `.form` · `.tag` |
| 품사 필터 | 명사·형용사·동사만 남김 | `tag.startswith(('NN','VA','VV'))` |
| 불용어 제거 | 뜻 없는 흔한 말 버리기 | 일반(`stopwords_ko.json`) **+ 도메인** |

- 한국어는 **조사·어미가 붙기 때문에** 공백 `split()` 만으로는 같은 단어가 갈라집니다 → **형태소 분석** 필요.
- 핵심 표현은 문자 클래스 `[...]`·부정 `[^...]`·수량자 `+`/`{m,n}`·시작/끝 `^`/`$`·그룹/OR·역참조 `\1`입니다.
- **도메인 불용어는 데이터를 봐야 압니다** — 빈도 상위를 확인하고 사람이 골라 목록을 만듭니다. 영화 리뷰의 `영화`·`평점` 이 그렇습니다.
- 완성한 **`clean_text` · `tokenize`** 가 앞으로의 모든 텍스트 분석의 **입구**입니다.

## ⏭️ 예고 — 다음 시간: 전통 텍스트 분석

이제 리뷰가 **깨끗한 단어 목록**으로 바뀌었습니다. 다음 시간엔 이 단어들로 실제 분석을 합니다.

- **단어 빈도**와 **워드클라우드** — 무슨 말이 많이 나왔는지 한눈에 보기
- **BoW · TF-IDF** — 문서를 숫자 표로 바꾸고, **이 문서에서만 중요한 핵심어** 뽑기
- **긍정/부정을 가르는 특징어** 추출 — 어떤 말이 좋은 평을, 어떤 말이 나쁜 평을 만드는가
- **n-gram**과 **동시 출현(연관 분석)** — 함께 붙어 다니는 단어 찾기

오늘 만든 `clean_text` · `tokenize` 를 그대로 씁니다. 수고하셨습니다!